## Cardiovascular Disease Prediction: Clinical ML for Early Detection¶

**Author**: Dean | Lead Data & AI Engineer
**Focus**: Healthcare ML for clinical decision support in government and enterprise health systems

---

Cardiovascular disease (CVD) remains the leading cause of death globally. Early prediction using routinely collected clinical data — blood pressure, cholesterol, BMI, lifestyle factors — can flag high-risk patients before a cardiac event occurs.

This notebook builds a **production-grade binary classifier** to predict cardiovascular disease presence from 70,000 patient records. The framing is clinical/enterprise:

- How does this integrate with **Electronic Health Records (EHR)** and **FHIR** standards?
- What do clinicians need from model outputs? (Not just a score — calibrated probabilities, risk tiers, explainable features.)
- How do we handle model fairness across demographic groups?
- How does this align with TGA (Therapeutic Goods Administration) requirements for AI/ML in Australian healthcare?


In [1]:
import os, warnings, time, glob
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve,
    precision_score, recall_score, brier_score_loss,
    log_loss
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.figsize': (12, 6), 'axes.titlesize': 14,
    'axes.labelsize': 12, 'font.size': 11, 'figure.dpi': 100,
})

C_HEALTHY = '#2ecc71'
C_DISEASE = '#e74c3c'
C_PALETTE = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12', '#9b59b6',
             '#1abc9c', '#e67e22', '#2c3e50']

print("Environment ready ✓")

Environment ready ✓


/home/ayo/myenv/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [2]:
df_raw = pd.read_csv("cardio_train.csv", sep=";")

In [3]:
df_raw.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


In [5]:
info_df = pd.DataFrame({
    'dtype': df_raw.dtypes,
    'non_null': df_raw.notnull().sum(),
    'null_pct': (df_raw.isnull().sum() / len(df_raw) * 100).round(2),
    'nunique': df_raw.nunique(),
    'min': df_raw.min(numeric_only=False),
    'max': df_raw.max(numeric_only=False),
})
info_df

,dtype,non_null,null_pct,nunique,min,max
id,int64,70000,0.0,70000,0.0,99999.0
age,int64,70000,0.0,8076,10798.0,23713.0
gender,int64,70000,0.0,2,1.0,2.0
height,int64,70000,0.0,109,55.0,250.0
weight,float64,70000,0.0,287,10.0,200.0
ap_hi,int64,70000,0.0,153,-150.0,16020.0
ap_lo,int64,70000,0.0,157,-70.0,11000.0
cholesterol,int64,70000,0.0,3,1.0,3.0
gluc,int64,70000,0.0,3,1.0,3.0
smoke,int64,70000,0.0,2,0.0,1.0


In [6]:
TARGET = 'cardio' if 'cardio' in df_raw.columns else df_raw.columns[-1]
print(f"Target: '{TARGET}'")
print(f"\nClass distribution:")
vc = df_raw[TARGET].value_counts()
for cls, cnt in vc.items():
    label = 'Healthy' if cls == 0 else 'CVD Present'
    print(f" {cls} ({label}): {cnt:,} ({cnt/len(df_raw)*100:.1f}%)")
    print(f"\n Balance ratio: {vc.min() / vc.max():.3f}")

Target: 'cardio'

Class distribution:
 0 (Healthy): 35,021 (50.0%)

 Balance ratio: 0.999
 1 (CVD Present): 34,979 (50.0%)

 Balance ratio: 0.999


#### 2 · Exploratory Data Analysis

Understanding the clinical profile of CVD patients is essential before modelling. Key questions:

- Demographics: What age/gender distribution do we see?
- Clinical measurements: How do blood pressure, cholesterol, and BMI differ between groups?
- Lifestyle factors: Do smoking, alcohol, and physical activity show clear associations?
- Data quality: Are there outliers (e.g., impossible blood pressure values)?


In [ ]:
if 'age' in df_raw.columns and df_raw['age'].max() > 200:
    df_raw['age_years'] = (df_raw['age'] / 365.25).round(1)
    print(f"Converted age from days to years: range [{df_raw['age_years'].min():.0f}, {df_raw['age_years'].max():.0f}")
    
# Blood pressure outliers (physiologically impossible values)

if 'ap_hi' in df_raw.columns:
    before = len(df_raw)
    
    # Remove extreme outliers
    df_clean = df_raw[
        (df_raw['ap_hi'] > 60) & (df_raw['ap_hi'] < 250) 
        &
        (df_raw['ap_lo'] > 30) & (df_raw['ap_lo'] < 200)
        &
        (df_raw['ap_hi'] > )
    ]